# PART 1
 getting blobs and ids from db connection

In [2]:
# trying to use the mudular code to get the workflow feeling
import sys
import json

# add app to path so we dont get the error ModuleNotFoundError: No module named 'app'
from pathlib import Path
sys.path.append(str(Path.cwd().parent))
from app.services.database.oracle import OracleService
from app.services.database.mariadb import MariaDBService
from app.utils.db_operations import load_query_from_file, execute_query_to_df
from app.utils.config import load_config
from app.utils.logger import get_logger


logger = get_logger(name=__name__)
config_vars = load_config()

In [2]:

aviso_cirurgia_query_file = "/home/joao/projects/company_projects/carteirinha-api/documents/querys/query_aviso_cirurgia.txt"
aviso_cirurgia_query = load_query_from_file(aviso_cirurgia_query_file)
query_file_path = "/home/joao/projects/company_projects/carteirinha-api/documents/querys/query_numero_carteirinha.txt"
numero_carteirinha_query = load_query_from_file(query_file_path)
query_file_path = "/home/joao/projects/company_projects/carteirinha-api/documents/querys/query_carteirinha.txt"
carteirinha_query = load_query_from_file(query_file_path)

In [4]:
maria_db_carteirinha_df = execute_query_to_df(db_service_class=MariaDBService(settings=config_vars), query=aviso_cirurgia_query, fetch_limit=500)
oracle_db_numero_carteirinha_df = execute_query_to_df(db_service_class=OracleService(settings=config_vars), query=numero_carteirinha_query)
oracle_db_carteirinha_df = execute_query_to_df(db_service_class=OracleService(settings=config_vars), query=carteirinha_query)


{"timestamp": "2025-08-15T14:48:36", "level": "INFO", "name": "app.services.database.mariadb", "message": "✅ MariaDB connection established to\n                srvawsdb002.cow7tj30bxpl.us-east-1.rds.amazonaws.com:3306/", "filename": "mariadb.py", "lineno": 27}
{"timestamp": "2025-08-15T14:48:36", "level": "INFO", "name": "app.services.database.mariadb", "message": "Executing MariaDB query...", "filename": "mariadb.py", "lineno": 67}
{"timestamp": "2025-08-15T14:48:45", "level": "INFO", "name": "app.services.database.mariadb", "message": "✅ Fetched 500 rows from MariaDB.", "filename": "mariadb.py", "lineno": 75}
{"timestamp": "2025-08-15T14:48:45", "level": "INFO", "name": "app.services.database.mariadb", "message": "✅ MariaDB connection closed.", "filename": "mariadb.py", "lineno": 41}
{"timestamp": "2025-08-15T14:48:45", "level": "INFO", "name": "app.utils.db_operations", "message": "✅ Query executed successfully with 500 sample records", "filename": "db_operations.py", "lineno": 57}


In [ ]:
# make some individual df processing if needed
maria_db_carteirinha_df.rename(columns={"surgical_order_id": "CD_AVISO_CIRURGIA", "health_insurance_name": "CONVENIO"}, inplace=True)

In [10]:
# merge the dfs inner join on aviso cirurgia
import pandas as pd

df_merged_oracle = pd.merge(oracle_db_carteirinha_df, oracle_db_numero_carteirinha_df, on="CD_AVISO_CIRURGIA", how="inner")
logger.info("✅ Inner join completed successfully.")

df_merged_final = pd.merge(df_merged_oracle, maria_db_carteirinha_df, on="CD_AVISO_CIRURGIA", how="inner")

df_merged_final.drop_duplicates(subset=["LO_DOCUMENTO_ANEXO_CIRURGICO"], inplace=True)
droppable_cols = [col for col in df_merged_final.columns if col not in ["CD_AVISO_CIRURGIA", "LO_DOCUMENTO_ANEXO_CIRURGICO", 'patient_name', 'CONVENIO', 'NR_CARTEIRA']]
df_merged_final.drop(columns=droppable_cols, inplace=True)
logger.info(df_merged_final.info())
df_merged_final.head()

{"timestamp": "2025-08-15T15:07:19", "level": "INFO", "name": "__main__", "message": "✅ Inner join completed successfully.", "filename": "3573579702.py", "lineno": 5}
{"timestamp": "2025-08-15T15:07:19", "level": "INFO", "name": "__main__", "message": "None", "filename": "3573579702.py", "lineno": 12}


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 226 entries, 0 to 225
Data columns (total 5 columns):
 #   Column                        Non-Null Count  Dtype 
---  ------                        --------------  ----- 
 0   CD_AVISO_CIRURGIA             226 non-null    int64 
 1   LO_DOCUMENTO_ANEXO_CIRURGICO  226 non-null    object
 2   NR_CARTEIRA                   226 non-null    object
 3   patient_name                  226 non-null    object
 4   CONVENIO                      226 non-null    object
dtypes: int64(1), object(4)
memory usage: 9.0+ KB


,CD_AVISO_CIRURGIA,LO_DOCUMENTO_ANEXO_CIRURGICO,NR_CARTEIRA,patient_name,CONVENIO
0,792045,b'%PDF-1.4\n1 0 obj\n<<\n/Title (\xfe\xff\x00D...,891722600159007,NILDA CRISTINA BRAGA RODRIGUES,BRADESCO
1,792058,b'%PDF-1.4\n1 0 obj\n<<\n/Title (\xfe\xff\x00D...,954560112208012,GABRIELLY CELIANA DE REZENDE,BRADESCO
2,792061,b'%PDF-1.4\n1 0 obj\n<<\n/Title (\xfe\xff\x00E...,88888483405330026,LETICIA SCHNEIDER RIBEIRO,SUL AMERICA
3,792062,b'%PDF-1.4\n1 0 obj\n<<\n/Title (\xfe\xff\x00D...,775045002283004,REGINA MARCIA ALVES DE OLIVEIRA,BRADESCO
4,792064,b'%PDF-1.4\n1 0 obj\n<<\n/Title (\xfe\xff\x00D...,772102060828002,VANESSA CRISTINA LEITE,BRADESCO


In [25]:
# save merged df
import sqlalchemy

# Create a database connection
engine = sqlalchemy.create_engine("sqlite:///gold_carteirinha_database.sqlite")


# Save the DataFrame to a SQLite table
df_merged_final.to_sql("carteirinha", engine, if_exists="replace", index=False)

# load 
df_merged_final = pd.read_sql("SELECT * FROM carteirinha", engine)

# part 1.5

In [7]:
# load the dataframe to memory
import sqlalchemy
import pandas as pd

# Create a database connection
engine = sqlalchemy.create_engine("sqlite:///gold_carteirinha_database.sqlite")
df_merged_final = pd.read_sql("SELECT * FROM carteirinha", engine)

In [8]:
# GETTING ids and blobs dict  like ids_and_blobs = {id: blob,id2: blob} from cd aviso and lo docuemnto anexo from df
def get_ids_and_blobs(df: pd.DataFrame) -> dict:
    ids_and_blobs = {}
    for index, row in df.iterrows():
        ids_and_blobs[row['CD_AVISO_CIRURGIA']] = row['LO_DOCUMENTO_ANEXO_CIRURGICO']
    return ids_and_blobs

ids_and_blobs = get_ids_and_blobs(df_merged_final)


# PART 2
 use texttrackt kv and llm reasoning strat to evaluate accuracy on numero carteirinha

In [9]:
# trying to use the mudular code to get the workflow feeling
import sys

# add app to path so we dont get the error ModuleNotFoundError: No module named 'app'
from pathlib import Path
sys.path.append(str(Path.cwd().parent))
from app.utils.process_images import process_blobs
from app.utils.textract_service import TextractKVExtractor, extract_text_single_id
from app.utils.llm_service import AnthropicLLMService
from app.services.database.oracle import OracleService
from app.services.database.mariadb import MariaDBService
from app.utils.db_operations import load_query_from_file, execute_query_to_df
from app.utils.aws_services_handler import create_boto3_client
from app.utils.config import load_config, AppConstants
from app.utils.logger import get_logger
from app.utils.system_prompts.prompt_handler import Prompts


logger = get_logger(name=__name__)
config_vars = load_config()

In [10]:
app_constants = AppConstants()
texttract_client = create_boto3_client("textract", config_vars)
bedrock_client = create_boto3_client("bedrock-runtime", config_vars)
carteirinha_prompt = Prompts.carteirinha_extraction_prompt

texttract_instance = TextractKVExtractor(texttract_client)
llm_instance = AnthropicLLMService(
    model_id=app_constants.BEDROCK_DEFAULT_MODEL_ID,
    model_version=app_constants.BEDROCK_DEFAULT_MODEL_VERSION,
    client=bedrock_client,
    system_prompt=carteirinha_prompt,
    max_tokens=app_constants.MAX_TOKENS,
    temperature=app_constants.TEMPERATURE,
    budget_tokens=app_constants.BUDGET_TOKENS
)


{"timestamp": "2025-08-18T09:27:10", "level": "INFO", "name": "app.utils.aws_services_handler", "message": "Criando cliente TEXTRACT para a região: us-east-1...", "filename": "aws_services_handler.py", "lineno": 17}
{"timestamp": "2025-08-18T09:27:10", "level": "INFO", "name": "app.utils.aws_services_handler", "message": "Cliente TEXTRACT criado com sucesso.", "filename": "aws_services_handler.py", "lineno": 26}
{"timestamp": "2025-08-18T09:27:10", "level": "INFO", "name": "app.utils.aws_services_handler", "message": "Criando cliente BEDROCK-RUNTIME para a região: us-east-1...", "filename": "aws_services_handler.py", "lineno": 17}
{"timestamp": "2025-08-18T09:27:10", "level": "INFO", "name": "app.utils.aws_services_handler", "message": "Cliente BEDROCK-RUNTIME criado com sucesso.", "filename": "aws_services_handler.py", "lineno": 26}


In [ ]:
# PROCESSING CUSTOM D=DICT WITH IDS AND BLOBS
byte_png_images_and_ids = process_blobs(ids_and_blobs, img_enhancement=False, MAX_IMAGES_PER_BLOB=20)
logger.info(f"Processed {len(byte_png_images_and_ids)} byte PNG images.")

In [ ]:
textract_results = {}
llm_results = {}
for id in byte_png_images_and_ids:
    single_id_full_text = extract_text_single_id(images_bytes_list=byte_png_images_and_ids[id],
                                                 textract_instance=texttract_instance, extract_full_text=False)

    textract_results[id] = single_id_full_text
    llm_response = llm_instance.invoke_model(input_str=single_id_full_text)
    llm_results[id] = llm_response

# save to df
textract_results_df = pd.DataFrame.from_dict(textract_results, orient="index")
textract_results_df.index.name = "CD_AVISO_CIRURGIA"
textract_results_df = textract_results_df.reset_index()

llm_results_df = pd.DataFrame.from_dict(llm_results, orient="index")
llm_results_df.index.name = "CD_AVISO_CIRURGIA"
llm_results_df = llm_results_df.reset_index()

# save to sql
import sqlalchemy

# Create a database connection
engine_textract = sqlalchemy.create_engine("sqlite:///textract_results_database.sqlite")
engine_llm = sqlalchemy.create_engine("sqlite:///llm_results_database.sqlite")

textract_results_df.to_sql("textract_results", engine_textract, if_exists="replace", index=False)
llm_results_df.to_sql("llm_results", engine_llm, if_exists="replace", index=False)


# Part 3:
## evaluation with generated predictions and actual values from db

In [ ]:
import sqlalchemy

# Create a database connection
engine_textract = sqlalchemy.create_engine("sqlite:///textract_results_database.sqlite")
engine_llm = sqlalchemy.create_engine("sqlite:///llm_results_database.sqlite")

textract_results_df = pd.read_sql("SELECT * FROM textract_results", engine_textract)
llm_results_df = pd.read_sql("SELECT * FROM llm_results", engine_llm)

In [ ]:
df_merged_final.drop_duplicates(subset=["LO_DOCUMENTO_ANEXO_CIRURGICO"], inplace=True)
df_merged_final.to_string()
llm_results_df.to_string()
eval_df = pd.merge(df_merged_final, llm_results_df, on="CD_AVISO_CIRURGIA", how="inner")
eval_df.head()